# Unitree G1 Colab GPU IK Benchmark

This notebook is a hardware-free Unitree G1 arm inverse-kinematics smoke test for teleoperation and retargeting work. It downloads the G1 URDF from `unitreerobotics/xr_teleoperate`, extracts the left and right torso-to-palm chains, solves batched wrist targets on a Colab GPU, and renders visual checks for error, workspace coverage, and joint-limit margin.

Run it top-to-bottom on a fresh GPU runtime. Treat a pass as evidence that the current model/parser/solver path is healthy, not as proof of robot safety.


## Notebook map and data-science practices

**Goal.** Validate a reproducible GPU IK benchmark for the Unitree G1 arms without requiring hardware.

**Inputs.** The upstream G1 URDF URL, fixed random seeds, batch size, optimization steps, and the Colab GPU runtime.

**Outputs.** Summary metric cards, error charts, 3D target-vs-solved plots, joint-limit heatmaps, and assertive pass/fail checks.

**Good notebook hygiene used here:**

- Keep the experiment description, assumptions, and limitations in Markdown before the code.
- Put configuration near the top so reruns are easy to audit.
- Keep reusable implementation in `src/unitree_colab_ik`; the notebook is an executable report.
- Record runtime versions and input provenance in the visible output.
- Use deterministic seeds for comparable benchmark runs.
- Prefer compact, named sections over long exploratory cells.
- Put metrics and visualizations next to interpretation text.
- End with assertions so a failed run is obvious.
- Restart the runtime and run all cells before publishing.

**Known limits.** This is a position-only smoke test. It does not validate wrist orientation, self-collision, external collision, torque limits, controller latency, network behavior, or hardware safety.


## References

- Unitree `xr_teleoperate`: https://github.com/unitreerobotics/xr_teleoperate
- Unitree G1 assets: https://github.com/unitreerobotics/xr_teleoperate/tree/main/assets/g1
- Google Colab FAQ: https://research.google.com/colaboratory/faq.html
- Project Jupyter documentation: https://docs.jupyter.org/en/latest/
- Ten simple rules for writing and sharing computational analyses in Jupyter Notebooks: https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1007007
- OpenAI Codex subagents: https://developers.openai.com/codex/concepts/subagents
- OpenAI harness engineering: https://openai.com/index/harness-engineering/


## 1. Setup

Clone the repository into ephemeral Colab storage and install it in editable mode. The notebook avoids writing to Google Drive so reruns do not mutate user files.


In [ ]:
REPO_URL = "https://github.com/zack-dev-cm/unitree-g1-colab-ik.git"
WORKDIR = "/content/unitree-g1-colab-ik"
BATCH_SIZE = 256
STEPS = 220
SUCCESS_THRESHOLD_M = 0.02

!rm -rf {WORKDIR}
!git clone --depth 1 {REPO_URL} {WORKDIR}
%pip -q install -e {WORKDIR}


## 2. Runtime guard

The benchmark is intended for a GPU runtime. This cell fails early if CUDA is unavailable and records the key runtime versions in the output.


In [ ]:
import torch
from IPython.display import HTML, display

assert torch.cuda.is_available(), "Enable Runtime > Change runtime type > GPU before running this notebook."
gpu_name = torch.cuda.get_device_name(0)
display(HTML(f"""
<div style="font-family:system-ui,-apple-system,Segoe UI,sans-serif;border:1px solid #d9e2ec;border-radius:10px;padding:16px 18px;background:#f8fbff;max-width:820px">
  <div style="font-size:13px;color:#486581;text-transform:uppercase;letter-spacing:.08em">Runtime</div>
  <div style="font-size:26px;font-weight:750;color:#102a43;margin-top:4px">{gpu_name}</div>
  <div style="display:flex;gap:16px;margin-top:10px;color:#334e68;font-size:14px;flex-wrap:wrap">
    <div><b>torch</b> {torch.__version__}</div>
    <div><b>cuda</b> {torch.version.cuda}</div>
    <div><b>batch</b> {BATCH_SIZE}</div>
    <div><b>steps</b> {STEPS}</div>
  </div>
</div>
"""))


## 3. Run the IK workload

For each arm, the benchmark builds a torso-to-palm chain from the Unitree G1 URDF, samples reachable wrist targets from bounded joint poses, and optimizes joint angles with PyTorch Adam. Targets are reachable by construction, so a failure points to solver, parser, dependency, or runtime regression.


In [ ]:
from unitree_colab_ik.benchmark import build_chain, load_robot, solve_tracking_ik

robot = load_robot()
details = {}
for index, side in enumerate(("left", "right")):
    chain = build_chain(robot, side, device="cuda")
    solved = solve_tracking_ik(
        chain,
        batch_size=BATCH_SIZE,
        steps=STEPS,
        seed=2026 + index,
        success_threshold_m=SUCCESS_THRESHOLD_M,
    )
    q = solved["q"].detach()
    errors = solved["errors"].detach()
    margin = torch.minimum(q - chain.lower, chain.upper - q) / chain.half_range
    details[side] = {
        "chain": chain,
        "q": q.cpu(),
        "target_positions": solved["target_positions"].cpu(),
        "final_positions": solved["final_positions"].cpu(),
        "errors": errors.cpu(),
        "margin": margin.detach().cpu(),
        "elapsed_s": solved["elapsed_s"],
        "success_rate": solved["success_rate"],
        "limit_violation_rad": solved["limit_violation_rad"],
    }

summary = []
for side, item in details.items():
    errors = item["errors"]
    summary.append({
        "side": side,
        "mean_cm": float(errors.mean() * 100),
        "p95_cm": float(torch.quantile(errors, 0.95) * 100),
        "max_cm": float(errors.max() * 100),
        "success_rate": item["success_rate"],
        "elapsed_s": item["elapsed_s"],
        "targets_per_s": BATCH_SIZE / item["elapsed_s"],
        "limit_violation_rad": item["limit_violation_rad"],
    })


## 4. Summary metrics

The primary acceptance signal is wrist position error under the 2 cm threshold with zero joint-limit violation. Throughput is included as a rough runtime-health signal, not as a production latency claim.


In [ ]:
cards = []
for row in summary:
    accent = "#1f9d8a" if row["side"] == "left" else "#d9480f"
    cards.append(f"""
    <div style="border:1px solid #d9e2ec;border-top:5px solid {accent};border-radius:10px;background:white;padding:15px;min-width:250px;box-shadow:0 3px 10px rgba(16,42,67,.06)">
      <div style="font-size:13px;color:#627d98;text-transform:uppercase;letter-spacing:.08em">{row['side']} arm</div>
      <div style="font-size:30px;font-weight:800;color:#102a43;margin:4px 0">{row['mean_cm']:.2f} cm</div>
      <div style="color:#486581">mean wrist position error</div>
      <hr style="border:none;border-top:1px solid #edf2f7;margin:12px 0">
      <div style="display:grid;grid-template-columns:1fr 1fr;gap:8px;font-size:13px;color:#334e68">
        <div><b>p95</b><br>{row['p95_cm']:.2f} cm</div>
        <div><b>max</b><br>{row['max_cm']:.2f} cm</div>
        <div><b>success</b><br>{row['success_rate']*100:.1f}%</div>
        <div><b>throughput</b><br>{row['targets_per_s']:.1f}/s</div>
      </div>
    </div>
    """)

display(HTML("""
<div style="font-family:system-ui,-apple-system,Segoe UI,sans-serif;max-width:980px">
  <h2 style="margin:0 0 10px;color:#102a43">Benchmark Summary</h2>
  <div style="display:flex;gap:16px;flex-wrap:wrap">%s</div>
</div>
""" % "".join(cards)))


## 5. Error distribution

The bars compare mean, p95, and worst-case error. The histograms make long tails visible, which is more useful than looking only at a mean.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#fbfdff",
    "axes.edgecolor": "#bcccdc",
    "axes.labelcolor": "#243b53",
    "axes.titleweight": "bold",
    "font.size": 11,
    "grid.color": "#d9e2ec",
})
colors = {"left": "#1f9d8a", "right": "#d9480f"}

labels = [row["side"] for row in summary]
x = np.arange(len(labels))
width = 0.24
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.bar(x - width, [row["mean_cm"] for row in summary], width, label="mean", color="#1f9d8a")
ax.bar(x, [row["p95_cm"] for row in summary], width, label="p95", color="#f59f00")
ax.bar(x + width, [row["max_cm"] for row in summary], width, label="max", color="#d9480f")
ax.axhline(SUCCESS_THRESHOLD_M * 100, color="#9b2c2c", linestyle="--", linewidth=1.3, label="2 cm target")
ax.set_xticks(x, labels)
ax.set_ylabel("position error (cm)")
ax.set_title("Unitree G1 wrist IK error by side")
ax.grid(axis="y", alpha=0.75)
ax.legend(frameon=False, ncols=4, loc="upper center", bbox_to_anchor=(0.5, 1.16))
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=3, fontsize=9)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8), sharey=True)
for ax, side in zip(axes, labels):
    err_cm = details[side]["errors"].numpy() * 100
    ax.hist(err_cm, bins=28, color=colors[side], alpha=0.88, edgecolor="white")
    ax.axvline(SUCCESS_THRESHOLD_M * 100, color="#9b2c2c", linestyle="--", linewidth=1.2)
    ax.set_title(f"{side.title()} arm error distribution")
    ax.set_xlabel("error (cm)")
    ax.grid(axis="y", alpha=0.6)
axes[0].set_ylabel("target count")
plt.tight_layout()
plt.show()


## 6. Spatial target check

Green points are target wrist positions and orange points are optimized wrist positions. Short connectors indicate small residuals; visible gaps would be a failure worth investigating.


In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

fig = plt.figure(figsize=(12, 5.2))
for plot_index, side in enumerate(labels, start=1):
    ax = fig.add_subplot(1, 2, plot_index, projection="3d")
    target = details[side]["target_positions"].numpy()
    solved = details[side]["final_positions"].numpy()
    sample = np.linspace(0, len(target) - 1, min(90, len(target)), dtype=int)
    ax.scatter(target[sample, 0], target[sample, 1], target[sample, 2], s=18, color="#1f9d8a", alpha=0.72, label="target")
    ax.scatter(solved[sample, 0], solved[sample, 1], solved[sample, 2], s=12, color="#d9480f", alpha=0.62, label="solved")
    for i in sample[::8]:
        ax.plot([target[i, 0], solved[i, 0]], [target[i, 1], solved[i, 1]], [target[i, 2], solved[i, 2]], color="#829ab1", alpha=0.35, linewidth=0.8)
    ax.set_title(f"{side.title()} wrist targets vs solved poses")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.set_zlabel("z (m)")
    ax.legend(frameon=False, loc="upper left")
    ax.view_init(elev=22, azim=-55 if side == "left" else -125)
plt.tight_layout()
plt.show()


## 7. Joint-limit margin

The mean-margin heatmap shows typical remaining range after optimization. The worst-case heatmap highlights joints that approach a limit and should be reviewed before making the target generator harder.


In [ ]:
joint_labels = [name.replace("left_", "").replace("right_", "").replace("_joint", "") for name in details["left"]["chain"].joint_names]
mean_margin = np.vstack([details[side]["margin"].mean(dim=0).numpy() for side in labels]) * 100
min_margin = np.vstack([details[side]["margin"].min(dim=0).values.numpy() for side in labels]) * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), constrained_layout=True)
for ax, matrix, title in zip(axes, [mean_margin, min_margin], ["Mean joint-limit margin", "Worst-case joint-limit margin"]):
    image = ax.imshow(matrix, cmap="YlGnBu", vmin=0, vmax=100, aspect="auto")
    ax.set_title(title)
    ax.set_yticks(np.arange(len(labels)), labels)
    ax.set_xticks(np.arange(len(joint_labels)), joint_labels, rotation=35, ha="right")
    for row in range(matrix.shape[0]):
        for col in range(matrix.shape[1]):
            ax.text(col, row, f"{matrix[row, col]:.0f}%", ha="center", va="center", color="#102a43", fontsize=9)
    ax.set_xlabel("G1 arm joint")
fig.colorbar(image, ax=axes, shrink=0.85, label="remaining range before limit (%)")
plt.show()


## 8. Validation and interpretation

The final cell converts the report into a pass/fail artifact. If it fails, keep the output and inspect the summary cards, distribution tails, and joint-limit heatmap before changing thresholds.


In [ ]:
for row in summary:
    assert row["success_rate"] >= 0.98, row
    assert row["mean_cm"] <= 1.0, row
    assert row["limit_violation_rad"] <= 1e-6, row

display(HTML("""
<div style="font-family:system-ui,-apple-system,Segoe UI,sans-serif;border-radius:10px;background:#edfdf7;border:1px solid #8eedc7;padding:16px 18px;max-width:820px;color:#014d40">
  <div style="font-size:24px;font-weight:800">PASS</div>
  <div style="margin-top:4px">Unitree G1 Colab GPU IK benchmark passed for both arms with zero joint-limit violations.</div>
  <div style="margin-top:9px;font-size:13px;line-height:1.5;color:#0b5d4f">Next validation layers: orientation targets, collision checks, latency measurement, and hardware-in-the-loop testing.</div>
</div>
"""))
